In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import glob
import os
from torch.utils.data import Subset

In [68]:
# -------------------------------
# 1️⃣ 读取谱线列表文件
# -------------------------------
def read_line_list(filepath):
    lines, freqs, mark07, mark09, mark10 = [], [], [], [], []
    with open(filepath, 'r') as f:
        
        # 跳过第一行
        next(f)
        
        for line in f:
            if line.strip() == '':
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            lines.append(parts[0])
            freqs.append(float(parts[1]))
            
            if len(parts) >= 5:
                mark07.append(int(parts[2]))
                mark09.append(int(parts[3]))
                mark10.append(int(parts[4]))
            else:
                mark07.append(0)
                mark09.append(0)
                mark10.append(0)
                
    return lines, np.array(freqs), mark07, mark09, mark10

# -------------------------------
# 2️⃣ 多文件光谱数据集类
# -------------------------------
class MultiSourceSpectrumDataset(Dataset):
    def __init__(self, data_dir, line_file, window=0.03, target_len=128,
                 normalize=True, velocity_dict=None):
        """
        初始化多源光谱数据集
        
        参数:
        - data_dir: 包含所有光谱文件的目录
        - line_file: 谱线列表文件路径
        - window: 提取窗口大小
        - target_len: 目标子光谱长度
        - normalize: 是否归一化
        - velocity_dict: 字典，键为源名称，值为速度修正值 (km/s)
        """
        self.line_names, self.line_rest, self.mark07, self.mark09, self.mark10 = read_line_list(line_file)
        self.line_name_to_index = {name: i for i, name in enumerate(self.line_names)}
        print(len(self.line_names))
        print(len(self.line_rest))
        
        self.marks_dict = {
        'Lh07': self.mark07,
        'Lh09': self.mark09,
        'Lh10': self.mark10,}
        
        print("----------------检查谱线标签----------------")
        print(self.marks_dict)
        print("--------------------------------------------")
        
        self.c = 299792458  # m/s
        self.window = window
        self.target_len = target_len
        self.normalize = normalize
        
        # 如果没有提供速度字典，创建空字典
        if velocity_dict is None:
            velocity_dict = {}
        
        # 存储所有样本和对应的信息
        self.samples = []        # 光谱数据
        self.full_names = []     # 完整标识符，格式: "源-频率窗口-谱线名称"
        self.source_info = []    # 源信息，格式: (源名称, 频率窗口, 谱线名称)
        self.scales=[]           # 初始化缩放因子列表
        self.marks=[]            # 谱线标签（该频率是否存在信号）
        
        # 查找所有光谱文件
        # 假设文件命名格式: "spectrum.源名称.频率窗口.commonbeam.1arcsec.dat"
        pattern = os.path.join(data_dir, "spectrum.*.spw*.commonbeam.1arcsec.dat")
        spectrum_files = glob.glob(pattern)
        
        if not spectrum_files:
            print(f"警告: 在目录 {data_dir} 中没有找到匹配的光谱文件")
            return
        
        print(f"找到 {len(spectrum_files)} 个光谱文件")
        
        # 处理每个光谱文件
        for spectrum_file in spectrum_files:
            # 从文件名中提取源名称和频率窗口
            filename = os.path.basename(spectrum_file)
            parts = filename.split('.')
            if len(parts) < 3:
                print(f"跳过无法解析的文件: {filename}")
                continue
                
            source_name = parts[1]  # 例如 "Lh07"
            spw = parts[2]          # 例如 "spw0"
            
            print(f"处理文件: {filename}, 源: {source_name}, 频率窗口: {spw}")
            
            # 获取该源的速度修正值
            velocity = velocity_dict.get(source_name, None)
            
            # 处理单个光谱文件
            self._process_single_file(spectrum_file, source_name, spw, velocity)
            
        print(f"总共提取了 {len(self.samples)} 个谱线样本")
        
#-------------------------------------寻找重复谱线-----------------------------------------
        print("寻找重复谱线")
        result = self._find_duplicate_c_by_a_simple(self.source_info)
        print("\n简洁版结果:")
        for a, c_dict in result.items():
            print(f"\n源 {a}:")
            for c, positions in c_dict.items():
                count = len(positions)
                print(f"  谱线 {c}: 重复{count}次，位置{positions}")
                
#-------------------------------------删除重复谱线------------------------------------------------

        # 收集所有要删除的位置（每个重复组的第二个位置）
        indices_to_remove = []
        for a, c_dict in result.items():
            for c, positions in c_dict.items():
                # 每个重复组取第二个位置（索引1）
                if len(positions) >= 2:
                    indices_to_remove.append(positions[1])

        # 从大到小排序，避免删除时索引变化
        indices_to_remove.sort(reverse=True)

        # 打印要删除的位置
        print(f"\n要删除的位置: {indices_to_remove}")

        # 删除这些位置的数据
        for idx in indices_to_remove:
            # 删除每个列表中的对应元素
            del self.samples[idx]
            del self.full_names[idx]
            del self.source_info[idx]
            del self.scales[idx]
            del self.marks[idx]

        print(f"删除后剩余样本数: {len(self.samples)}")

#-----------------------------------------------------------------------------
    
    
            
#-----------------------找哪条谱线重复了----------------------------------------
    def _find_duplicate_c_by_a_simple(self,data):
        """
        简洁版本：找出当a相同时，重复的c
        返回: {a: {重复的c: [位置列表]}}
        """
        from collections import defaultdict

        # 构建字典: a -> c -> [位置列表]
        a_c_positions = defaultdict(lambda: defaultdict(list))

        # 一次遍历收集所有信息
        for idx, item in enumerate(data):
            a = item[0]  # source_name
            c = item[3]  # line_name (第四个元素，频率)
            a_c_positions[a][c].append(idx)

        # 筛选出有重复c的a
        result = {}
        for a, c_dict in a_c_positions.items():
            duplicates = {c: positions for c, positions in c_dict.items() if len(positions) > 1}
            if duplicates:
                result[a] = duplicates

        return result

#-----------------------找哪条谱线重复了----------------------------------------


#---------------------------处理单个光谱文件---------------------------------
    def _process_single_file(self, spectrum_file, source_name, spw, velocity=None):
#         try:
            
        # 读取主光谱，跳过第一行
        data = np.loadtxt(spectrum_file, skiprows=1)
        freq_obs, flux = data[:, 0], data[:, 1]
        

        # 速度校正
        if velocity is not None:
            v = velocity * 1e3  # km/s → m/s
            freq = freq_obs / (1 - v / self.c)
            print(f"  [INFO] 应用速度修正: v = {velocity:.3f} km/s")
        else:
            freq = freq_obs

        print("开始提取子光谱")

        print(f"总共有{len(self.line_names)}条光谱")
        print("------------------------------")
        number=0
        # 对每条谱线提取子光谱
        for i in range(len(self.line_names)):
            line_name = self.line_names[i]
            f_rest = self.line_rest[i]
            
            mask = (freq > f_rest - self.window) & (freq < f_rest + self.window)
#             print(i)
#             print(f"检查第{i}条曲线，为{line_name},静止频率为{f_rest}")
            if np.sum(mask) < 8:
                continue  # 跳过数据点太少的谱线

            f_sub = freq[mask]
            flux_sub = flux[mask]
            mark = self.marks_dict[source_name][i]
            print(f"提取第{i}条曲线，为{line_name},静止频率为{f_rest}，长度为{len(flux_sub)}, 信号标签为{mark}")
            

            
            # 转换到速度区间
            v_sub = ((f_sub-f_rest)/f_rest)*self.c/1000
#             print(line_name)
#             plt.plot(v_sub,flux_sub)
#             plt.show()
    
#             plt.plot(f_sub,flux_sub)
#             plt.show()
            
            # 把不完整的谱线去掉
            if max(v_sub)<20.0 or min(v_sub>-20.0):
                print("------不完整，删掉------")
                continue
            number+=1
    
            # 插值到目标长度
#             f_new = np.linspace(f_sub.min(), f_sub.max(), self.target_len)
            v_new = np.linspace(-27.0, 27.0, self.target_len)
            flux_new = np.interp(v_new, v_sub, flux_sub)
            
#             plt.plot(flux_new)
#             plt.show()
            
            # 归一化 (可选)
            if self.normalize:
                #flux_new = (flux_new - np.mean(flux_new)) / (np.std(flux_new) + 1e-6)
                scale = max(np.max(np.abs(flux_new)), 1e-6)
                flux_norm = flux_new / scale
            else:
                scale=1.0
                flux_norm=flux_new


        
            # 创建完整标识符
            full_name = f"{source_name}-{spw}-{line_name}"

            # 保存样本和相关信息
            self.samples.append(torch.tensor(flux_norm, dtype=torch.float32).unsqueeze(0))
            self.full_names.append(full_name)
            self.source_info.append((source_name, spw, line_name, f_rest))
            self.scales.append(scale)  # 保存了缩放因子
            
            # 保存谱线标签（是否存在信号）
            self.marks.append(mark)

        print(f"这个文件共提取了{number}条谱线")

#     except Exception as e:
#         print(f"处理文件 {spectrum_file} 时出错: {e}")

#---------------------------处理单个光谱文件---------------------------------


    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx], self.full_names[idx], self.scales[idx]
    
    def get_mark_by_line(self, idx=None, source_name=None, line_name=None):

    #获取特定谱线的标记（是否存在信号）

        # 如果提供了索引，获取源名称和谱线名称
        if idx is not None:
            if idx >= len(self.source_info):
                print(f"警告: 索引 {idx} 超出范围")
                return None
            source_name, _, line_name = self.source_info[idx]

        # 验证参数
        if source_name is None or line_name is None:
            print("错误: 必须提供 source_name 和 line_name，或者提供 idx")
            return None

        # 检查源是否有对应的标记数据
        if source_name not in self.marks_dict:  # 使用 marks_dict
            print(f"警告: 源 {source_name} 没有标记数据")
            return None

        # 获取谱线索引
        if line_name not in self.line_name_to_index:
            print(f"警告: 谱线 {line_name} 不存在")
            return None

        line_idx = self.line_name_to_index[line_name]

        # 获取标记值
        marks_list = self.marks_dict[source_name]  # 从 marks_dict 获取
        return marks_list[line_idx]
    
    def get_source_info(self, idx):
        """获取指定索引样本的源信息"""
        return self.source_info[idx]
    
    def get_samples_by_source(self, source_name):
        """获取特定源的所有样本索引"""
        return [i for i, info in enumerate(self.source_info) if info[0] == source_name]
    
    def get_samples_by_spw(self, spw):
        """获取特定频率窗口的所有样本索引"""
        return [i for i, info in enumerate(self.source_info) if info[1] == spw]
    
    def get_samples_by_line(self, line_name):
        """获取特定谱线的所有样本索引"""
        return [i for i, info in enumerate(self.source_info) if info[2] == line_name]


In [69]:
def collate_fn(batch):
    """处理批次数据，包括缩放因子"""
    xs = torch.stack([b[0] for b in batch])  # 光谱数据
    names = [b[1] for b in batch]            # 名称列表
    
    scales_list = [b[2] for b in batch]
    scales = torch.tensor(scales_list, dtype=torch.float32)
    scales = scales.view(-1, 1, 1)  # 重塑为 [batch_size, 1, 1]
    
    return xs, names, scales

In [70]:
if __name__ == "__main__":
    # 数据目录和谱线列表
    data_dir = "C:\\Users\\zyx\\Desktop\\Spectral with Machine Learning\\data\\manysource"
    line_file = "C:\\Users\\zyx\\Desktop\\Spectral with Machine Learning\\data\\linelist_with_mark.txt"
    
    # 定义各源的速度修正值 (km/s)
    velocity_dict = {
        "Lh07": 239.5157166,
        "Lh09": 235.751358,  
        "Lh10": 251.2273865,  
        # 添加更多源和对应的速度...
    }
    
    # 创建多源数据集
    dataset = MultiSourceSpectrumDataset(
        data_dir=data_dir,
        line_file=line_file,
        velocity_dict=velocity_dict,
        target_len=256,
        normalize=True
    )
    

65
65
----------------检查谱线标签----------------
{'Lh07': [1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0], 'Lh09': [1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], 'Lh10': [1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0]}
--------------------------------------------
找到 15 个光谱文件
处理文件: spectrum.Lh07.spw0.commonbeam.1arcsec.dat, 源: Lh07, 频率窗口: spw0
  [INFO] 应用速度修正: v = 239.516 km/s
开始提取子光谱
总共有65条光谱
------------------------------
提取第5条曲线，为34SO2,静止频率为344.2453476，长度为122, 信号标签为1
提取第6条曲线，为34SO2,静止频率为344.581045，长度为123, 信号标签为1
提取第7条曲线，为34SO2,静止频率为344.8079157，长度为122

In [72]:
# 新建一个类保存上面的dataset
class create_data_Results:
    def __init__(self):
        self.dataset = dataset

In [77]:
# mark1=dataset.marks
# positions = [i for i, value in enumerate(mark1) if value == 1]
# print(len(positions))
# print(positions)

In [75]:
# for i in range(len(positions)):
#     n=positions[i]
#     plt.plot(dataset[n][0].squeeze().numpy())
#     plt.show()

In [43]:
# subset = Subset(dataset,dataset.get_samples_by_source("Lh07"))

In [76]:
# for i in range(len(subset)):
#     print(subset[i][1])
#     plt.plot(subset[i][0].squeeze().numpy())
#     plt.show()